# PPLSTM — T4 fused CUDA kernel experiment (verification-loop fix v2)

This revision fixes the `KeyError(8)` caused by an undefined batch-size entry for the eight-token numerical test. It retains the original PPLSTM equations and CUDA kernel source unchanged. GPU execution and numerical equivalence must be verified on Colab.

## 1 · Connect Colab GPU and configure a short run

In Colab: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Installation/compilation can take longer than the actual timed benchmark. If GPU unavailable, the first code cell gives an actionable error. This notebook never accesses Kaggle.

In [ ]:
import sys, os, subprocess, importlib.util, time, json, csv, zipfile, math, copy, random, traceback, statistics
from pathlib import Path
SEED=1907
WIDTH=72
HEADS=4
CONTEXTS=(128,512)
BATCHES={128:8,512:4}
WARMUP=3
REPEATS=12
BENCHMARK_GRAPHS=True
BENCHMARK_FULL_HYBRID=True
REQUIRE_NUMERIC_CHECKS=True
OUTPUT_DIR=Path('/content/pplstm_t4_fused_results' if Path('/content').exists() else '/mnt/data/pplstm_t4_fused_results')
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

def pip_missing(module, package):
    if importlib.util.find_spec(module) is None:
        print('Installing',package,flush=True)
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
pip_missing('xlstm','xlstm==2.0.6')
pip_missing('cupy','cupy-cuda12x')

import numpy as np, torch, cupy as cp
from torch import nn
from torch.nn import functional as F
assert torch.cuda.is_available(), 'No NVIDIA GPU found. Colab: Runtime → Change runtime type → T4 GPU, then reconnect and Run all.'
DEVICE=torch.device('cuda')
CAPABILITY=torch.cuda.get_device_capability()
print('GPU:',torch.cuda.get_device_name(0),'CC',CAPABILITY,'PyTorch',torch.__version__,'CUDA',torch.version.cuda)
print('CuPy',cp.__version__)
assert CAPABILITY[0]>=7, 'This prototype targets NVIDIA T4 (CC 7.5) or newer.'
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32=False
torch.set_num_threads(2)
CHECKS=[]; RESULTS=[]; NOTES=[]
def check(name, passed, **details):
    row={'check':name,'pass':bool(passed),**details}; CHECKS.append(row)
    print(('PASS' if passed else 'FAIL'),name,details,flush=True)
def skip(name,exc):
    NOTES.append({'name':name,'error':repr(exc)[:2000]})
    print('SKIPPED',name,repr(exc)[:500],flush=True)


## 2 · Your original PPLSTM reference (unchanged)

This is copied directly from your prior Colab notebook, which itself reproduced your published [`pplstm.py`](https://github.com/rickey1990/PPLSTM/blob/main/pplstm.py). The fused test inherits these parameters and `_token_maps` without editing their equations.

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
# Adapted from the original PPLSTM project by Rickey (https://github.com/rickey1990/PPLSTM/blob/main/pplstm.py)
# The transition equations and scan implementation are kept as published.
"""PPLSTM: associative 3-channel permutation-mixed recurrent cell."""
import itertools
import math
import torch
from torch import Tensor, nn
from torch.nn import functional as F


def shift_right(x: Tensor) -> Tensor:
    return torch.cat((torch.zeros_like(x[:, :1]), x[:, :-1]), dim=1)


def scalar_affine_scan(a: Tensor, b: Tensor) -> tuple[Tensor, Tensor]:
    A, B = a, b
    step = 1
    while step < a.shape[1]:
        A, B = (
            torch.cat((A[:, :step], A[:, step:] * A[:, :-step]), dim=1),
            torch.cat((B[:, :step], B[:, step:] + A[:, step:] * B[:, :-step]), dim=1),
        )
        step *= 2
    return A, B


def matrix_affine_scan(A: Tensor, b: Tensor) -> tuple[Tensor, Tensor]:
    step = 1
    while step < A.shape[1]:
        next_A = torch.cat((A[:, :step], A[:, step:] @ A[:, :-step]), dim=1)
        next_b = torch.cat((
            b[:, :step],
            b[:, step:] + (A[:, step:] @ b[:, :-step].unsqueeze(-1)).squeeze(-1),
        ), dim=1)
        A, b = next_A, next_b
        step *= 2
    return A, b


class PPLSTM(nn.Module):
    def __init__(self, width: int):
        super().__init__()
        if width < 3 or width % 3 != 0:
            raise ValueError('width must be a positive multiple of 3')
        self.width = width
        self.groups = width // 3
        self.xproj = nn.Linear(width, 4 * width)
        self.mix_logit = nn.Parameter(torch.full((4, width), math.log(3.0)))
        self.sign_proj = nn.Linear(width, width)
        self.initial_cell = nn.Parameter(torch.linspace(-1.5, 1.5, 3).repeat(self.groups))
        self.input_bias = nn.Parameter(torch.full((width,), -4.0))
        self.forget_bias = nn.Parameter(torch.full((width,), 4.0))
        nn.init.zeros_(self.sign_proj.weight)
        nn.init.constant_(self.sign_proj.bias, 2.8)

        bank = []
        for order in itertools.permutations(range(3)):
            matrix = torch.zeros(3, 3)
            for row, column in enumerate(order):
                matrix[row, column] = 1.0
            bank.append(matrix)
        self.register_buffer('permutation_bank', torch.stack(bank))

        self.selector = nn.Linear(width, self.groups * 6)
        nn.init.zeros_(self.selector.weight)
        nn.init.constant_(self.selector.bias, -1.0)
        with torch.no_grad():
            for group in range(self.groups):
                self.selector.bias[group * 6] = 3.0

    def _token_maps(self, x: Tensor):
        if x.ndim != 3 or x.shape[-1] != self.width:
            raise ValueError('Expected x with shape [batch, time, width]')
        batch, length, width = x.shape
        group_count = self.groups
        previous = shift_right(x)
        coeff = self.mix_logit.sigmoid().view(1, 1, 4, width)
        gates = (self.xproj(x).reshape(batch, length, 4, width) * coeff
                 + self.xproj(previous).reshape(batch, length, 4, width) * (1 - coeff))
        input_logits, forget_logits, candidate_logits, output_logits = gates.unbind(2)
        mixed = x * self.mix_logit[0].sigmoid() + previous * (1 - self.mix_logit[0].sigmoid())
        signs = self.sign_proj(mixed).tanh().reshape(batch, length, group_count, 3)
        z = candidate_logits.tanh().reshape(batch, length, group_count, 3)
        log_i = (input_logits + self.input_bias).reshape(batch, length, group_count, 3).mean(-1)
        log_f = F.logsigmoid((forget_logits + self.forget_bias)
                           .reshape(batch, length, group_count, 3).mean(-1))
        f_prefix = log_f.cumsum(dim=1)
        m = f_prefix + torch.cummax(torch.cat((
            torch.zeros_like(f_prefix[:, :1]), log_i - f_prefix,
        ), dim=1), dim=1).values[:, 1:]
        f = (log_f + shift_right(m) - m).exp()
        i = (log_i - m).exp()
        weights = self.selector(mixed).reshape(batch, length, group_count, 6).softmax(-1)
        P = torch.einsum('btgp,pij->btgij', weights, self.permutation_bank)
        A = f[..., None, None] * P * signs[..., None, :]
        b = i[..., None] * z
        return A, b, f, i, output_logits

    def forward(self, x: Tensor) -> Tensor:
        A, b, f, i, output_logits = self._token_maps(x)
        batch, length, _ = x.shape
        prefix_A, prefix_b = matrix_affine_scan(A, b)
        c0 = self.initial_cell.tanh().reshape(1, self.groups, 3).expand(batch, -1, -1)
        c = (prefix_A @ c0[:, None, :, :, None]).squeeze(-1) + prefix_b
        norm_A, norm_b = scalar_affine_scan(f, i)
        n = norm_A + norm_b
        ratio = (c / n[..., None].clamp_min(1e-12)).reshape(batch, length, self.width)
        return output_logits.sigmoid() * ratio

    def serial_reference(self, x: Tensor) -> Tensor:
        A, b, f, i, output_logits = self._token_maps(x)
        batch, length, _ = x.shape
        c = self.initial_cell.tanh().reshape(1, self.groups, 3).expand(batch, -1, -1)
        n = x.new_ones(batch, self.groups)
        states = []
        for t in range(length):
            c = (A[:, t] @ c.unsqueeze(-1)).squeeze(-1) + b[:, t]
            n = f[:, t] * n + i[:, t]
            states.append(c / n[..., None].clamp_min(1e-12))
        ratio = torch.stack(states, dim=1).reshape(batch, length, self.width)
        return output_logits.sigmoid() * ratio


## 3 · Compile **two CUDA kernels** for the associative scan and its backward

A CUDA thread block handles one batch × three-channel memory group. Each thread handles a time position; shared memory implements a parallel doubling scan. We combine matrix `A,b` and scalar `f,i` scans in the same stages and write the final gated/normalized output. The backward kernel uses a reverse parallel affine scan to obtain the memory and normalizer adjoints. CUDA Graph replay is measured separately.

**T4 implementation limit:** length ≤512 and FP32 inputs; this is a diagnostic baseline rather than a production kernel. Source is saved in the exported ZIP for inspection and future optimization.

In [ ]:
CUDA_SOURCE = '\n// NVRTC: no host C math.h header needed; expf/fmaxf are CUDA device built-ins.\n\n// One CTA per (batch, group), one thread per token. Elements beyond T are identity maps.\n// Memory: A_t (3x3), b_t (3), f_t, i_t. Fused scan output is normalized and gated.\nextern "C" __global__ void pplstm_fused_fwd(\n const float *__restrict__ A, const float *__restrict__ b,\n const float *__restrict__ f, const float *__restrict__ i,\n const float *__restrict__ o, const float *__restrict__ c0,\n float *__restrict__ h, float *__restrict__ cstate, float *__restrict__ nstate,\n int T, int G) {\n  const int lane=threadIdx.x;\n  const int group=blockIdx.x % G;\n  const int batch=blockIdx.x / G;\n  const int gidx=(batch*T+lane)*G+group;\n  const int bidx=gidx*3, aidx=gidx*9;\n  float aa[9], bb[3];\n  float ff=1.f, ii=0.f;\n  #pragma unroll\n  for(int j=0;j<9;++j) aa[j]=(j%4==0)?1.f:0.f;\n  bb[0]=0.f;bb[1]=0.f;bb[2]=0.f;\n  if(lane<T) {\n    #pragma unroll\n    for(int j=0;j<9;++j) aa[j]=A[aidx+j];\n    #pragma unroll\n    for(int j=0;j<3;++j) bb[j]=b[bidx+j];\n    ff=f[gidx];ii=i[gidx];\n  }\n  extern __shared__ float sh[];\n  for(int step=1;step<blockDim.x;step<<=1) {\n    float *own=sh+lane*14;\n    #pragma unroll\n    for(int j=0;j<9;++j) own[j]=aa[j];\n    #pragma unroll\n    for(int j=0;j<3;++j) own[9+j]=bb[j];\n    own[12]=ff;own[13]=ii;\n    __syncthreads();\n    if(lane>=step) {\n      const float *prev=sh+(lane-step)*14;\n      float na[9],nb[3];\n      #pragma unroll\n      for(int row=0;row<3;++row) {\n        #pragma unroll\n        for(int col=0;col<3;++col) {\n          na[row*3+col]=aa[row*3]*prev[col]\n                         +aa[row*3+1]*prev[3+col]\n                         +aa[row*3+2]*prev[6+col];\n        }\n        nb[row]=bb[row]+aa[row*3]*prev[9]\n                           +aa[row*3+1]*prev[10]\n                           +aa[row*3+2]*prev[11];\n      }\n      float next_ii=ii+ff*prev[13];\n      ff=ff*prev[12];ii=next_ii;\n      #pragma unroll\n      for(int j=0;j<9;++j) aa[j]=na[j];\n      #pragma unroll\n      for(int j=0;j<3;++j) bb[j]=nb[j];\n    }\n    __syncthreads();\n  }\n  if(lane<T) {\n    const int out=(batch*T+lane)*(G*3)+group*3;\n    const int start=(batch*G+group)*3;\n    float nn=fmaxf(ff+ii,1.e-12f);\n    nstate[gidx]=ff+ii;\n    #pragma unroll\n    for(int j=0;j<3;++j) {\n      float cc=bb[j]+aa[j*3]*c0[start]\n                     +aa[j*3+1]*c0[start+1]\n                     +aa[j*3+2]*c0[start+2];\n      cstate[bidx+j]=cc;\n      float sig=1.f/(1.f+expf(-o[out+j]));\n      h[out+j]=sig*(cc/nn);\n    }\n  }\n}\n\n// Reverse affine scan for the state and normalizer adjoints.\n// q_t = dL/dc_t + A_(t+1)^T q_(t+1),\n// u_t = dL/dn_t + f_(t+1) u_(t+1).\nextern "C" __global__ void pplstm_fused_bwd(\n const float *__restrict__ A, const float *__restrict__ f,\n const float *__restrict__ o, const float *__restrict__ c0,\n const float *__restrict__ cstate, const float *__restrict__ nstate,\n const float *__restrict__ gh,\n float *__restrict__ ga, float *__restrict__ gb,\n float *__restrict__ gf, float *__restrict__ gi,\n float *__restrict__ go, float *__restrict__ gc0,\n int T, int G) {\n  const int lane=threadIdx.x;\n  const int t=T-1-lane;\n  const int group=blockIdx.x%G, batch=blockIdx.x/G;\n  float aa[9],bb[3],ff=1.f,ii=0.f;\n  #pragma unroll\n  for(int j=0;j<9;++j) aa[j]=(j%4==0)?1.f:0.f;\n  bb[0]=0.f;bb[1]=0.f;bb[2]=0.f;\n  if(t>=0) {\n    int gidx=(batch*T+t)*G+group;\n    int bidx=gidx*3;\n    int out=(batch*T+t)*(G*3)+group*3;\n    float nn=nstate[gidx];\n    float denom=fmaxf(nn,1.e-12f);\n    float dn=0.f;\n    #pragma unroll\n    for(int j=0;j<3;++j) {\n      float sig=1.f/(1.f+expf(-o[out+j]));\n      float grad=gh[out+j];\n      float cc=cstate[bidx+j];\n      bb[j]=grad*sig/denom;\n      dn-=grad*sig*cc/(denom*denom);\n      go[out+j]=grad*(cc/denom)*sig*(1.f-sig);\n    }\n    ii=(nn>1.e-12f)?dn:0.f;\n    if(t+1<T) {\n      ff=f[(batch*T+t+1)*G+group];\n      int nextA=((batch*T+t+1)*G+group)*9;\n      #pragma unroll\n      for(int row=0;row<3;++row)\n        #pragma unroll\n        for(int col=0;col<3;++col)\n          aa[row*3+col]=A[nextA+col*3+row]; // transpose\n    }\n  }\n  extern __shared__ float sh[];\n  for(int step=1;step<blockDim.x;step<<=1) {\n    float *own=sh+lane*14;\n    #pragma unroll\n    for(int j=0;j<9;++j) own[j]=aa[j];\n    #pragma unroll\n    for(int j=0;j<3;++j) own[9+j]=bb[j];\n    own[12]=ff;own[13]=ii;\n    __syncthreads();\n    if(lane>=step) {\n      const float *prev=sh+(lane-step)*14;\n      float na[9],nb[3];\n      #pragma unroll\n      for(int row=0;row<3;++row) {\n        #pragma unroll\n        for(int col=0;col<3;++col)\n          na[row*3+col]=aa[row*3]*prev[col]\n                         +aa[row*3+1]*prev[3+col]\n                         +aa[row*3+2]*prev[6+col];\n        nb[row]=bb[row]+aa[row*3]*prev[9]\n                           +aa[row*3+1]*prev[10]\n                           +aa[row*3+2]*prev[11];\n      }\n      float new_ii=ii+ff*prev[13];\n      ff=ff*prev[12];ii=new_ii;\n      #pragma unroll\n      for(int j=0;j<9;++j) aa[j]=na[j];\n      #pragma unroll\n      for(int j=0;j<3;++j) bb[j]=nb[j];\n    }\n    __syncthreads();\n  }\n  if(t>=0) {\n    int gidx=(batch*T+t)*G+group;\n    int bidx=gidx*3;\n    int aidx=gidx*9;\n    int start=(batch*G+group)*3;\n    float prevc[3], prevn;\n    if(t==0) {\n      prevc[0]=c0[start];prevc[1]=c0[start+1];prevc[2]=c0[start+2];\n      prevn=1.f;\n    } else {\n      int previous=(batch*T+t-1)*G+group;\n      prevc[0]=cstate[previous*3];\n      prevc[1]=cstate[previous*3+1];\n      prevc[2]=cstate[previous*3+2];\n      prevn=nstate[previous];\n    }\n    #pragma unroll\n    for(int row=0;row<3;++row) {\n      gb[bidx+row]=bb[row];\n      #pragma unroll\n      for(int col=0;col<3;++col)\n        ga[aidx+row*3+col]=bb[row]*prevc[col];\n    }\n    gf[gidx]=ii*prevn;\n    gi[gidx]=ii;\n    if(t==0) {\n      #pragma unroll\n      for(int col=0;col<3;++col) {\n        gc0[start+col]=A[aidx+col]*bb[0]\n                       +A[aidx+3+col]*bb[1]\n                       +A[aidx+6+col]*bb[2];\n      }\n    }\n  }\n}\n'
# CuPy NVRTC compiles the CUDA code on the Colab GPU; it is not a simulated kernel.
(OUTPUT_DIR/'pplstm_t4_fused_kernels.cu').write_text(CUDA_SOURCE)
try:
    KERNEL_FWD=cp.RawKernel(CUDA_SOURCE,'pplstm_fused_fwd',options=('--std=c++11',),backend='nvrtc')
    KERNEL_BWD=cp.RawKernel(CUDA_SOURCE,'pplstm_fused_bwd',options=('--std=c++11',),backend='nvrtc')
    # Force compile before CUDA Graph capture and outside the measured steps.
    KERNEL_FWD.compile()
    KERNEL_BWD.compile()
    KERNEL_READY=True
    print('Compiled both CUDA kernels successfully for',torch.cuda.get_device_name(0))
except Exception as exc:
    KERNEL_READY=False
    skip('compile_cuda_kernels',exc)


## 4 · Autograd wrapper and reference-only numerical tests

Before timing, compare the CUDA kernel to the original PyTorch cell with **identical weights and inputs**, including all parameter gradients and input gradients. CUDA operation may round differently; the test reports absolute and relative errors. Failure means the optimized path is **not trusted or used in performance claims**.

In [ ]:
def _cuda_arrays(*tensors):
    # torch.Tensor implements the CUDA DLPack protocol; no copies of GPU data.
    return [cp.from_dlpack(t.detach()) for t in tensors]

class FusedAffinePPLSTM(torch.autograd.Function):
    @staticmethod
    def forward(ctx,A,b,f,i,o,c0):
        assert KERNEL_READY, 'CUDA kernel did not compile'
        assert all(x.dtype==torch.float32 and x.is_cuda and x.is_contiguous() for x in (A,b,f,i,o,c0))
        B,T,G=A.shape[:3]
        assert 1<=T<=512,'T4 prototype currently supports length <=512'
        threads=1<<(T-1).bit_length()
        h=torch.empty((B,T,G*3),device=A.device,dtype=torch.float32)
        c=torch.empty_like(b)
        n=torch.empty_like(f)
        stream=cp.cuda.ExternalStream(torch.cuda.current_stream().cuda_stream)
        with stream:
            KERNEL_FWD((B*G,),(threads,),tuple(_cuda_arrays(A,b,f,i,o,c0,h,c,n))+ (np.int32(T),np.int32(G)),
                       shared_mem=threads*14*4,stream=stream)
        ctx.save_for_backward(A,f,o,c0,c,n)
        ctx.B,ctx.T,ctx.G=B,T,G
        return h
    @staticmethod
    def backward(ctx,dh):
        A,f,o,c0,c,n=ctx.saved_tensors
        B,T,G=ctx.B,ctx.T,ctx.G
        dh=dh.contiguous()
        ga=torch.empty_like(A);gb=torch.empty_like(c)
        gf=torch.empty_like(f);gi=torch.empty_like(f)
        go=torch.empty_like(o);gc0=torch.empty_like(c0)
        threads=1<<(T-1).bit_length()
        stream=cp.cuda.ExternalStream(torch.cuda.current_stream().cuda_stream)
        with stream:
            KERNEL_BWD((B*G,),(threads,),tuple(_cuda_arrays(A,f,o,c0,c,n,dh,ga,gb,gf,gi,go,gc0))+
                       (np.int32(T),np.int32(G)),shared_mem=threads*14*4,stream=stream)
        return ga,gb,gf,gi,go,gc0

class T4FusedPPLSTM(PPLSTM):
    # NO modifications to PPLSTM parameters, _token_maps, initialization, routing or gates.
    # Replaces only scan + initial memory application + normalizer + final output.
    def forward(self,x):
        A,b,f,i,o=self._token_maps(x)
        B,T,_=x.shape
        c0=self.initial_cell.tanh().reshape(1,self.groups,3).expand(B,-1,-1).contiguous()
        return FusedAffinePPLSTM.apply(A.contiguous(),b.contiguous(),f.contiguous(),
                                     i.contiguous(),o.contiguous(),c0)

def error_stats(ref,new):
    delta=(ref.detach().float()-new.detach().float())
    denom=ref.detach().float().norm().clamp_min(1e-8)
    return {'max_abs':float(delta.abs().max()),'relative_l2':float(delta.norm()/denom)}

def compare_reference_fused(width,length,batch=2):
    torch.manual_seed(14000+length)
    old=PPLSTM(width).to(DEVICE)
    new=T4FusedPPLSTM(width).to(DEVICE)
    new.load_state_dict(old.state_dict(),strict=True)
    inp=torch.randn(batch,length,width,device=DEVICE)
    x1=inp.detach().requires_grad_(True)
    x2=inp.detach().requires_grad_(True)
    weights=torch.randn_like(inp)
    y1=old(x1); y2=new(x2)
    l1=(y1*weights).mean()+y1.square().mean()*0.17
    l2=(y2*weights).mean()+y2.square().mean()*0.17
    l1.backward();l2.backward()
    torch.cuda.synchronize()
    out=error_stats(y1,y2)
    igrad=error_stats(x1.grad,x2.grad)
    byparam={name:error_stats(dict(old.named_parameters())[name].grad, p.grad)
             for name,p in new.named_parameters()}
    worst_abs=max(v['max_abs'] for v in byparam.values())
    worst_rel=max(v['relative_l2'] for v in byparam.values())
    # Conservative pilot tolerances. Report all comparisons; never hide near misses.
    ok=(out['max_abs']<5e-4 and out['relative_l2']<5e-3
        and igrad['max_abs']<1e-3 and igrad['relative_l2']<0.03
        and worst_abs<1e-3 and worst_rel<0.03)
    check(f'original_vs_fused_width{width}_length{length}',ok,output=out,
          input_gradient=igrad,worst_parameter_gradient_max_abs=worst_abs,
          worst_parameter_gradient_relative_l2=worst_rel,per_parameter=byparam)
    return ok

# First verify reference cell scan/serial in float64 (independent of CUDA kernel).
check_model=PPLSTM(12).to('cpu',dtype=torch.float64)
check_in=torch.randn(2,11,12,dtype=torch.float64)
with torch.no_grad():
    original_difference=float((check_model(check_in)-check_model.serial_reference(check_in)).abs().max())
check('reference_serial_parallel_float64',original_difference<1e-10,max_abs=original_difference)

KERNEL_VALID=False
if KERNEL_READY:
    try:
        flags=[compare_reference_fused(WIDTH,T,max(2,BATCHES.get(T,4)//2)) for T in (8,128,512)]
        KERNEL_VALID=all(flags)
    except Exception as exc:
        skip('numerical_verification',exc)
        traceback.print_exc()
if not KERNEL_VALID:
    print('WARNING: FUSED OPTIMIZATION NOT VERIFIED. Do not use for training; baseline tests still available.')
else:
    print('All direct forward/backward correctness tests PASS.')


## 5 · Build the original official mLSTM + PPLSTM hybrid

Only two PPLSTM cells are changed. Both mLSTM blocks, the convolutions, residual paths, normalization, feed-forward layers and all model weights remain the same. A training-step speedup here is more meaningful than timing the isolated scan only.

In [ ]:
from xlstm import (xLSTMBlockStack,xLSTMBlockStackConfig,mLSTMBlockConfig,
                   mLSTMLayerConfig,sLSTMBlockConfig,sLSTMLayerConfig,FeedForwardConfig)
class PPLSTMReplacementLayer(nn.Module):
    def __init__(self,original_layer, fused=False):
        super().__init__()
        self.config=original_layer.config
        self.conv1d=original_layer.conv1d if self.config.conv1d_kernel_size>0 else None
        self.conv_act_fn=original_layer.conv_act_fn if self.conv1d is not None else None
        self.group_norm=original_layer.group_norm
        self.dropout=original_layer.dropout
        self.pplstm=(T4FusedPPLSTM if fused else PPLSTM)(self.config.embedding_dim)
    def forward(self,x,**kwargs):
        B,S,W=x.shape
        conv_x=self.conv_act_fn(self.conv1d(x)) if self.conv1d is not None else x
        y=self.dropout(self.pplstm(conv_x))
        H=self.config.num_heads
        y=y.reshape(B,S,H,W//H).transpose(1,2)
        y=self.group_norm(y).transpose(1,2).reshape(B,S,W)
        return y

def make_hybrid(context):
    cfg=xLSTMBlockStackConfig(
        mlstm_block=mLSTMBlockConfig(mlstm=mLSTMLayerConfig(conv1d_kernel_size=4,
             qkv_proj_blocksize=4,num_heads=HEADS)),
        slstm_block=sLSTMBlockConfig(slstm=sLSTMLayerConfig(backend='vanilla',
             num_heads=HEADS,conv1d_kernel_size=4,bias_init='powerlaw_blockdependent'),
             feedforward=FeedForwardConfig(proj_factor=1.3,act_fn='gelu')),
        context_length=context,num_blocks=4,embedding_dim=WIDTH,slstm_at=[1,3])
    model=xLSTMBlockStack(cfg)
    for idx in [1,3]:model.blocks[idx].xlstm=PPLSTMReplacementLayer(model.blocks[idx].xlstm,fused=False)
    return model.to(DEVICE)

def replace_pplstm_with_fused(model):
    for idx in (1,3):
        existing=model.blocks[idx].xlstm
        new=T4FusedPPLSTM(existing.pplstm.width).to(DEVICE)
        new.load_state_dict(existing.pplstm.state_dict(),strict=True)
        existing.pplstm=new
    return model

HYBRID_VALID=False
if KERNEL_VALID:
    try:
        hybrid_flags=[]
        for length in CONTEXTS:
            old=make_hybrid(length)
            new=replace_pplstm_with_fused(copy.deepcopy(old))
            assert list(old.state_dict())==list(new.state_dict())
            torch.manual_seed(777+length)
            base=torch.randn(2,length,WIDTH,device=DEVICE)
            target=torch.randn_like(base)
            x1=base.detach().requires_grad_(True)
            x2=base.detach().requires_grad_(True)
            yy=old(x1);zz=new(x2)
            ((yy-target).square().mean()).backward()
            ((zz-target).square().mean()).backward()
            torch.cuda.synchronize()
            full_diff=error_stats(yy,zz)
            x_diff=error_stats(x1.grad,x2.grad)
            p1=dict(old.named_parameters());p2=dict(new.named_parameters())
            assert list(p1)==list(p2)
            grad_diff={k:error_stats(p1[k].grad,p2[k].grad) for k in p1}
            worst_grad=max(v['relative_l2'] for v in grad_diff.values())
            passes=(full_diff['max_abs']<1e-3 and full_diff['relative_l2']<0.005 and
                    x_diff['relative_l2']<0.03 and worst_grad<0.03)
            hybrid_flags.append(passes)
            check(f'full_hybrid_outputs_and_gradients_length{length}',passes,
                  output=full_diff,input_grad=x_diff,
                  worst_param_relative_l2=worst_grad,
                  worst_param_max_abs=max(v['max_abs'] for v in grad_diff.values()))
            del old,new,base,target,x1,x2,yy,zz
        HYBRID_VALID=all(hybrid_flags)
    except Exception as exc:skip('full_hybrid_forward_backward_verification',exc)
print('FULL HYBRID VERIFIED:',HYBRID_VALID)


## 6 · Measure fused vs original: eager and CUDA Graph full training computations

These measurements include **forward + backward**, not an optimizer step (to isolate the kernels). The mLSTM blocks do **not** change between configurations. CUDA events measure GPU time; attempts with failed numerical verification are labelled and **not claimed as successful optimizations**. Random inputs mean this notebook is short and independent of FineWeb-Edu.

In [ ]:
def zero_grads(model):
    for p in model.parameters():
        if p.grad is not None:p.grad.zero_()

def timed(model,shape,mode,label,context):
    B,T,W=shape
    x=torch.randn(B,T,W,device=DEVICE)
    model.train()
    def eager_run():
        zero_grads(model)
        model(x).square().mean().backward()
    if mode=='graph':
        side=torch.cuda.Stream(device=DEVICE)
        side.wait_stream(torch.cuda.current_stream())
        with torch.cuda.stream(side):
            for _ in range(3):eager_run()
        torch.cuda.current_stream().wait_stream(side)
        model.zero_grad(set_to_none=True)
        graph=torch.cuda.CUDAGraph()
        with torch.cuda.graph(graph):
            model(x).square().mean().backward()
        torch.cuda.synchronize()
        def run():
            zero_grads(model)
            graph.replay()
    else:
        for _ in range(3):eager_run()
        run=eager_run
    for _ in range(WARMUP):run()
    torch.cuda.synchronize()
    durations=[]
    for _ in range(REPEATS):
        a=torch.cuda.Event(enable_timing=True);b=torch.cuda.Event(enable_timing=True)
        a.record();run();b.record();b.synchronize()
        durations.append(a.elapsed_time(b))
    ms=float(statistics.median(durations))
    result={'context':T,'batch':B,'model':label,'mode':mode,'ms':ms,
            'tokens_per_sec':B*T*1000/ms,'width':W,
            'metric':'GPU forward/backward/gradient-zero, optimizer excluded',
            'params':sum(p.numel() for p in model.parameters()),'steps':REPEATS}
    RESULTS.append(result)
    print(f'{label:24s} {mode:6s} len={T:<3} {ms:8.3f} ms {result["tokens_per_sec"]:,.0f} tok/s')
    return result

def safe_bench(model,shape,mode,label,context):
    try:return timed(model,shape,mode,label,context)
    except Exception as exc:skip(label+' '+mode+' '+str(context),exc);return None

for T in CONTEXTS:
    B=BATCHES[T]
    print('\n============',T,'tokens',B,'batch ============')
    reference=make_hybrid(T)
    # Copying before graph capture ensures strictly matched parameters at start.
    isolated_ref=copy.deepcopy(reference.blocks[1])
    if KERNEL_VALID:
        fused_hybrid=replace_pplstm_with_fused(copy.deepcopy(reference))
        isolated_fused=copy.deepcopy(fused_hybrid.blocks[1])
    else:
        fused_hybrid=None;isolated_fused=None
    variants=[('PPLSTM_block_original',isolated_ref),('hybrid_original',reference)]
    if KERNEL_VALID:
        variants.append(('PPLSTM_block_fused',isolated_fused))
        if HYBRID_VALID:variants.append(('hybrid_fused',fused_hybrid))
    for label,model in variants:
        for mode in (('eager','graph') if BENCHMARK_GRAPHS else ('eager',)):
            candidate=copy.deepcopy(model)
            safe_bench(candidate,(B,T,WIDTH),mode,label,T)
            del candidate
    del reference,isolated_ref,variants
    if fused_hybrid is not None:del fused_hybrid,isolated_fused
    torch.cuda.empty_cache()


## 7 · Results, speedups and small downloadable ZIP

For each length/mode, compare the original and the CUDA kernel **at equal model shape and weights**. If a kernel failed compilation or numerical checks, it will not be presented as a valid optimization. CPU launch/compile time is excluded from CUDA-event timings. The source `.cu` file is included in the ZIP.

In [ ]:
import pandas as pd
if RESULTS:
    frame=pd.DataFrame(RESULTS).sort_values(['context','model','mode'])
    display(frame[['context','batch','model','mode','ms','tokens_per_sec','params']])
    frame.to_csv(OUTPUT_DIR/'gpu_timings.csv',index=False)
    SPEEDUPS=[]
    for T in CONTEXTS:
        for mode in ('eager','graph'):
            for which in ('PPLSTM_block','hybrid'):
                x=frame[(frame.context==T)&(frame.model==which+'_original')&(frame['mode']==mode)]
                y=frame[(frame.context==T)&(frame.model==which+'_fused')&(frame['mode']==mode)]
                if len(x)==1 and len(y)==1:
                    ratio=float(x.iloc[0]['ms']/y.iloc[0]['ms'])
                    SPEEDUPS.append({'context':T,'mode':mode,'part':which,
                                     'speedup_original_div_fused':ratio,
                                     'original_ms':float(x.iloc[0]['ms']),
                                     'fused_ms':float(y.iloc[0]['ms'])})
    if SPEEDUPS:
        speed=pd.DataFrame(SPEEDUPS)
        display(speed)
        speed.to_csv(OUTPUT_DIR/'fused_speedups.csv',index=False)
else:print('No GPU timing results; check setup and kernel checks above.')
if CHECKS:pd.DataFrame(CHECKS).to_csv(OUTPUT_DIR/'numerical_checks.csv',index=False)
if NOTES:pd.DataFrame(NOTES).to_csv(OUTPUT_DIR/'failures_and_skips.csv',index=False)
summary=dict(gpu=torch.cuda.get_device_name(0),capability=CAPABILITY,
             pytorch=torch.__version__,cupy=cp.__version__,kernel_compiled=KERNEL_READY,
             kernel_validated=KERNEL_VALID,full_hybrid_validated=HYBRID_VALID,checks=CHECKS,benchmarks=RESULTS,skips=NOTES,
             caution='T4 experimental FP32 parallel scan. Forward and backward numerical checks gated benchmark. GPU timing excludes optimizer and dataset. Long-run loss equivalence not tested.')
(OUTPUT_DIR/'summary.json').write_text(json.dumps(summary,indent=2,default=str))
archive=OUTPUT_DIR/'PPLSTM_T4_Fused_Scan_Short_Test_Results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUTPUT_DIR.glob('*.csv'))+sorted(OUTPUT_DIR.glob('*.json'))+sorted(OUTPUT_DIR.glob('*.cu')):
        z.write(p,p.name)
print('RESULT ZIP:',archive,'bytes:',archive.stat().st_size)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:print('Download from:',archive)


## Reading the results

- If **`fused_speedups.csv` >1**, the fused version is faster for that mode; <1 means slower. Always prefer **`hybrid / graph`** for the closest comparison to your Kaggle setup.
- If a **numerical check fails**, don't integrate the kernel; the test correctly reports the failure. A CUDA kernel may be mathematically equivalent but need numerical tuning.
- This prototype fuses the memory scan + normalizer + output, but leaves `_token_maps` (selector, six-way permutation mixing, signed carry, exponential stabilizer) in ordinary PyTorch. Profiling after this experiment can tell us whether those operations are the next candidates for fusion.
- The test measures full forward and backward **without an optimizer**. Your 200M FineWeb benchmark has embeddings, output head, AdamW and data movement, which will change end-to-end throughput.
- A separate source `.cu` is provided in the ZIP. This is a **test kernel**, not a fully validated training backend. If it wins and passes checks on your T4, we can subsequently add more aggressive fusion and longer contexts without altering the original PPLSTM cell.